# 15: Indicadores do Banco Central (a régua da renda fixa)

Um dividend yield de 8% é bom? Um retorno de 15% no ano foi muito ou pouco? Sozinhos, esses números não dizem.
O investidor brasileiro sempre tem uma alternativa sem risco de bolsa: deixar o dinheiro na renda fixa, rendendo
perto do CDI. Todo número de ação faz sentido **comparado com essa régua**, e o relatório do pipeline não a tinha.

O projeto já tinha a tool `busca_indicador_bcb` desde o começo, mas nenhum agente a usava. E ela tinha um erro:
o CDI vinha da série 12 do Banco Central, que é o CDI **de um dia**, e saía rotulado como taxa ao ano. Um
relatório diria "CDI de 0,05% a.a." quando o certo é perto de 14% ao ano.

Agora a SELIC, o CDI e o IPCA de 12 meses entram em toda análise, buscados **em código** pelo nó do YFinance (não
pelo LLM, que às vezes chamaria a tool e às vezes não) e anexados à mensagem dele, como os sinais de anomalia.

Este notebook não chama nenhum LLM: a API do Banco Central, a Brapi e o yfinance são gratuitos.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**SGS (Sistema Gerenciador de Séries Temporais):** o serviço do Banco Central que publica milhares de séries
econômicas, cada uma com um número. A API é pública, sem chave: `api.bcb.gov.br/dados/serie/bcdata.sgs.<numero>`.

**SELIC meta (série 432):** a taxa básica de juros, definida pelo Copom (Comitê de Política Monetária) a cada 45
dias, em % ao ano. É o principal instrumento do Banco Central contra a inflação: juro alto encarece o crédito,
esfria o consumo e segura os preços. A data que a série devolve é a da meta **vigente**, e pode estar à frente de
hoje.

**CDI (Certificado de Depósito Interbancário):** a taxa dos empréstimos de um dia entre bancos. Ela acompanha a
SELIC de perto, um pouco abaixo, porque os bancos podem aplicar no título público que rende a SELIC: ninguém
empresta a outro banco por muito menos do que isso. É a referência da renda fixa privada ("este CDB paga 100% do
CDI").

**Série diária x anualizada:** o CDI é apurado **por dia útil**. A série 12 traz a taxa de um dia (perto de 0,05%
ao dia). A série 4389 traz a mesma taxa anualizada na base de 252 dias úteis: (1 + taxa do dia)^252 - 1. As duas
dizem a mesma coisa em unidades diferentes. Confundir as duas foi o erro da tool antiga.

**IPCA (Índice Nacional de Preços ao Consumidor Amplo):** a inflação oficial do Brasil, medida pelo IBGE. A série
433 traz a variação de **um mês** e a 13522, o acumulado em **12 meses**, que é o número comparável com uma taxa
ao ano.

**Juro real:** quanto o dinheiro rende acima da inflação. A conta rápida é SELIC menos IPCA. A conta exata é
(1 + juro) / (1 + inflação) - 1. Um cuidado: o IPCA de 12 meses olha para trás (o que já aconteceu), enquanto a
SELIC meta é a taxa de hoje. O juro real assim calculado é uma aproximação, não uma previsão.

**Dividend yield (DY) comparado com o CDI:** o DY é o dividendo pago nos últimos 12 meses dividido pelo preço da
ação. Comparar com o CDI responde: só pelos dividendos, a ação rende mais ou menos que a renda fixa? Não é a
conta inteira (a ação também sobe e cai de preço, e os dividendos variam de um ano para outro), mas é a primeira
régua.

## 2. A tool, série por série

`busca_indicador_bcb` aceita só os indicadores do enum `IndicadorEconomico`, o que impede pedir uma série
qualquer. Cada chamada devolve um dicionário com o valor, a data e a unidade.

In [2]:
from fluxo_financeiro.shared.types import IndicadorEconomico
from fluxo_financeiro.tools.financial_data import busca_indicador_bcb

dados = {}
for indicador in IndicadorEconomico:
    dados[indicador] = busca_indicador_bcb.invoke({"input_data": {"indicador": indicador.value}})
    d = dados[indicador]
    if "error" in d:
        print(f"{indicador.name:10} (série {indicador.value}): {d['error']}")
    else:
        print(f"{indicador.name:10} (série {indicador.value:5}): {d['valor']:>7.2f} {d['unidade']:15} em {d['data']}")

2026-11-08 14:19:48,002 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=SELIC_META


2026-11-08 14:19:48,470 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: ok — SELIC_META=13.75% a.a.


2026-11-08 14:19:48,475 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=IPCA


SELIC_META (série   432):   13.75 % a.a.          em 09/12/2026


2026-11-08 14:19:49,100 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: ok — IPCA=0.82% a.m.


2026-11-08 14:19:49,103 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=CDI


... (2 outputs left out)

2026-11-08 14:19:49,497 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=IPCA_12M


CDI        (série  4389):   13.65 % a.a.          em 05/11/2026


2026-11-08 14:19:49,856 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: ok — IPCA_12M=4.58% em 12 meses


IPCA_12M   (série 13522):    4.58 % em 12 meses   em 01/09/2026


## 3. O CDI diário e o anualizado, lado a lado

A série 12 saiu do enum (ninguém precisa do CDI de um dia), então a célula abaixo a busca direto na API, só para a
comparação. Compondo a taxa de um dia por 252 dias úteis, chega-se perto do valor da série 4389.

In [3]:
import requests

url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados/ultimos/1?formato=json"
cdi_dia = requests.get(url, timeout=10).json()[0]
taxa_dia = float(cdi_dia["valor"])
composto = ((1 + taxa_dia / 100) ** 252 - 1) * 100

print(f"Série 12   (CDI do dia):        {taxa_dia:.6f}% ao dia, em {cdi_dia['data']}")
print(f"Composto por 252 dias úteis:    {composto:.2f}% ao ano")
cdi = dados[IndicadorEconomico.CDI]
if "error" not in cdi:
    print(f"Série 4389 (CDI anualizado):    {cdi['valor']:.2f}% ao ano, em {cdi['data']}")

Série 12   (CDI do dia):        0.050788% ao dia, em 05/11/2026
Composto por 252 dias úteis:    13.65% ao ano
Série 4389 (CDI anualizado):    13.65% ao ano, em 05/11/2026


## 4. O bloco que entra na análise

`indicadores_economicos_em_texto()` (sem `@tool`) busca SELIC, CDI e IPCA de 12 meses e monta o bloco que o
`yfinance_node` anexa à mensagem do YFinance. Cada série falha sozinha: se o Banco Central não responder uma
delas, a linha some. Se não responder nenhuma, o bloco vem vazio e a análise segue sem ele.

In [4]:
from fluxo_financeiro.tools.financial_data import indicadores_economicos_em_texto

print(indicadores_economicos_em_texto())

2026-11-08 14:19:51,165 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=SELIC_META


2026-11-08 14:19:51,578 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: ok — SELIC_META=13.75% a.a.


2026-11-08 14:19:51,581 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=CDI


2026-11-08 14:19:51,972 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: ok — CDI=13.65% a.a.


2026-11-08 14:19:51,976 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: indicador=IPCA_12M


2026-11-08 14:19:52,333 - fluxo_financeiro - INFO - [tool] busca_indicador_bcb: ok — IPCA_12M=4.58% em 12 meses


INDICADORES ECONÔMICOS (Banco Central, buscados em código):
- SELIC (meta): 13.75% a.a. (BCB, 09/12/2026)
- CDI: 13.65% a.a. (BCB, 05/11/2026)
- IPCA acumulado em 12 meses: 4.58% (BCB, 01/09/2026)


## 5. Juro real com os números de hoje

In [5]:
selic = dados[IndicadorEconomico.SELIC_META]["valor"]
ipca_12m = dados[IndicadorEconomico.IPCA_12M]["valor"]

aproximado = selic - ipca_12m
exato = ((1 + selic / 100) / (1 + ipca_12m / 100) - 1) * 100
print(f"SELIC meta {selic:.2f}% a.a. | IPCA de 12 meses {ipca_12m:.2f}%")
print(f"Juro real aproximado (SELIC - IPCA):            {aproximado:.2f}% a.a.")
print(f"Juro real exato ((1 + SELIC) / (1 + IPCA) - 1): {exato:.2f}% a.a.")

SELIC meta 13.75% a.a. | IPCA de 12 meses 4.58%
Juro real aproximado (SELIC - IPCA):            9.17% a.a.
Juro real exato ((1 + SELIC) / (1 + IPCA) - 1): 8.77% a.a.


## 6. O DY da VALE3 contra o CDI

O DY vem da linha `DY:` da tool do YFinance (`buscar_dados_acao_b3`, a mesma que o agente chama). Se o yfinance
não informar o DY, a linha vem com `N/A` e a comparação não é feita.

In [6]:
import re

from fluxo_financeiro.tools.financial_data import buscar_dados_acao_b3

saida = buscar_dados_acao_b3.invoke("VALE3")
achado = re.search(r"^DY:\s+([\d.]+)%", saida, re.MULTILINE)
cdi_ano = dados[IndicadorEconomico.CDI].get("valor")

if achado and cdi_ano:
    dy = float(achado.group(1))
    print(f"DY da VALE3: {dy:.2f}% | CDI: {cdi_ano:.2f}% a.a.")
    print(f"O DY equivale a {dy / cdi_ano * 100:.0f}% do CDI")
else:
    print("DY ou CDI indisponível: comparação não feita")

2026-11-08 14:19:52,346 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-08 14:19:54,838 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ok (2434 chars)


DY da VALE3: 6.82% | CDI: 13.65% a.a.
O DY equivale a 50% do CDI


## O que observar

- A data da SELIC meta (09/12/2026) está à frente de hoje: é a data até a qual a meta vigente vale, e o bloco mostra como o Banco Central manda, sem tratar como erro.
- A série 12 deu cerca de 0,05% ao dia, e compor por 252 dias úteis chegou exatamente ao valor da série 4389 (13,65% ao ano). A tool antiga mostrava o primeiro número com o rótulo do segundo.
- O CDI ficou um pouco abaixo da SELIC meta (13,65% contra 13,75%), como esperado: as duas andam juntas.
- O IPCA do mês (0,82%) e o de 12 meses (4,58%) são números de natureza diferente. Multiplicar o do mês por 12 não dá o anual, porque um mês de inflação alta não se repete necessariamente. Por isso o bloco usa a série 13522.
- O juro real aproximado (9,17%) passa do exato (8,77%) em 0,4 ponto. Com juro e inflação altos, a diferença entre as duas contas cresce.
- O DY da VALE3 ficou perto da metade do CDI. Só pelos dividendos, a ação rendeu menos que a renda fixa no último ano. Isso não fecha a conta (falta a variação do preço), mas é a comparação que o relatório agora consegue fazer, porque o CDI está na mensagem do YFinance.
- Os números são de quando o notebook rodou. As séries mudam: a SELIC a cada reunião do Copom, o CDI a cada dia útil e o IPCA uma vez por mês.